# IndoML Datathon — Track 1: Noise Event Detection (Vaani)

**Frozen BEATs + FDY-CRNN, mean-teacher semi-supervision on a per-tier masked loss,
frequency MixStyle for domain shift, cSEBBs post-processing.**

Run top to bottom. Set the runtime to **GPU** first:
`Runtime -> Change runtime type -> T4 GPU`.

| Step | Cell | Notes |
|---|---|---|
| 0 | GPU check | fails loudly if you forgot to switch runtime |
| 1 | Clone + install | never installs torch (Colab's is CUDA-linked) |
| 2 | Download data | pulls every shard from Hugging Face onto the VM |
| 3 | BEATs weights | ~360 MB, cached |
| 4 | Train | checkpoints to Drive if mounted |
| 5 | Tune cSEBBs | no retraining, biggest ROI per minute |
| 6 | Predict | writes `submission.json` |


## 0 · GPU check


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'
print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '|', torch.cuda.get_device_name(0))


## 1 · Clone the repo and install dependencies

`requirements.txt` deliberately excludes torch: Colab ships a CUDA-linked build and
pip-installing torch here would replace it with a CPU wheel and silently kill the GPU.


In [ ]:
REPO = 'https://github.com/raut7218/vaani-sed-track1.git'
import os, sys
if not os.path.exists('/content/vaani-sed-track1'):
    !git clone -q $REPO /content/vaani-sed-track1
%cd /content/vaani-sed-track1
!git pull -q || true

# everything except torch/torchaudio, which Colab already has
!pip install -q soundfile librosa 'datasets>=2.18' huggingface_hub pyyaml tqdm
sys.path.insert(0, '/content/vaani-sed-track1')
print('ready')


### Optional · mount Drive so checkpoints survive a disconnect


In [ ]:
USE_DRIVE = False  # set True to persist data + checkpoints

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/vaani_track1'
else:
    BASE = '/content/work'
import os; os.makedirs(BASE, exist_ok=True)
DATA = f'{BASE}/data'; RUN = f'{BASE}/runs/baseline'
print('DATA =', DATA, '\nRUN  =', RUN)


## 2 · Download the dataset from Hugging Face

This pulls **every** parquet shard published in the repo
([PavanKumarJ-ARTPARK/Vaani_Noise_Event_TimeStamp](https://huggingface.co/datasets/PavanKumarJ-ARTPARK/Vaani_Noise_Event_TimeStamp))
straight onto the Colab VM — nothing is read from your local machine — decodes each
clip to audio, and writes `manifest.jsonl` with the tier assigned per clip.

Both stages are **resumable**: the HF cache skips shards already fetched and the
materialiser skips clips already in the manifest. Re-run it whenever new batches
are published.


First, see what the server actually holds — this downloads nothing:


In [ ]:
!python scripts/download_data.py --out $DATA --list-only


Now download and materialise everything:


In [ ]:
# --format flac is lossless and about half the size of wav; use it if DATA
# points at Google Drive. Add --token hf_... to raise the HF rate limit.
!python scripts/download_data.py --out $DATA --format flac

import json
print(json.dumps(json.load(open(f'{DATA}/stats.json')), indent=2)[:2000])


> **If the coverage line reports far less than ~167 h**, that is the dataset, not
> this code: the remainder has not been uploaded to Hugging Face yet. The script
> tells you exactly how many shards exist and how large they are. Re-run it as
> batches land — it resumes and only fetches what is new.

*(`src/data/prepare.py` is the older equivalent that goes through
`datasets.load_dataset`. It still works; `download_data.py` is preferred because it
shows you the real file list and streams shards with flat memory.)*


### Tier assignment — read this

The released parquet has **no tier column**. `prepare.py` resolves tiers as:

* **bronze** — no `NoiseSubCategoryTimeStamp` entries (this is exactly the bronze definition)
* **gold / silver** — needs a signal the data does not yet expose, so timestamped clips
  default to `silver` (the conservative choice: silver gets a lower strong-loss weight).

If the organisers publish gold ids, drop them in a text file and rerun with
`--gold-ids gold.txt`; if they add a tier column, `prepare.py` picks it up automatically
(it checks `tier`, `quality`, `annotation_quality`, `verified`, `num_annotators`, ...).

To treat all timestamped clips as gold instead: `--default-ts-tier gold`.


## 3 · BEATs checkpoint (~360 MB, cached)


In [ ]:
from src.models.beats_encoder import download_beats
p = download_beats('checkpoints')
print('BEATs at:', p)


## 4 · Train

One model, all three tiers. Every batch mixes gold/silver/bronze so mean-teacher
consistency and per-tier MixStyle always have material to work with.

On a T4, start with `--batch-size 16` if you hit OOM.


In [ ]:
!python -m src.train.train \
    --config configs/default.yaml \
    --data $DATA \
    --out $RUN \
    --epochs 30 \
    --batch-size 24


### Training curve


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'{RUN}/history.json'))
fig, ax = plt.subplots(figsize=(7,4))
for which in ('student','teacher'):
    xs = [r['epoch'] for r in h if r['which']==which]
    ys = [r['score'] for r in h if r['which']==which]
    if xs: ax.plot(xs, ys, marker='o', label=which)
ax.set_xlabel('epoch'); ax.set_ylabel('0.5*F1 + 0.5*Dice'); ax.legend(); ax.grid(alpha=.3)
plt.show()
print('best:', max((r['score'] for r in h), default=None))


## 5 · Tune the post-processor

Runs on cached validation scores — **no retraining**. This is the cheapest large win
available; it also prints the plain median-filter baseline so you can see the delta.


In [ ]:
!python scripts/tune_postproc.py --run $RUN --rounds 2


## 6 · Predict → `submission.json`

Point `--audio-dir` at the released test audio. Output is class-agnostic
onset/offset pairs, which is what Track 1 is scored on.


In [ ]:
TEST_AUDIO = '/content/test_audio'   # <-- point at the official test set

import os
if os.path.isdir(TEST_AUDIO) and os.listdir(TEST_AUDIO):
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --audio-dir $TEST_AUDIO \
        --params $RUN/postproc_params.json \
        --out submission.json
else:
    print('No test audio yet — running on the training manifest as a demo instead.')
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --manifest $DATA/manifest.jsonl \
        --params $RUN/postproc_params.json \
        --out submission.json


In [ ]:
import json
sub = json.load(open('submission.json'))
print(len(sub), 'clips,', sum(len(v) for v in sub.values()), 'events')
for uid in list(sub)[:3]:
    print(uid, sub[uid][:4])


### Download the submission


In [ ]:
from google.colab import files
files.download('submission.json')


---
## Where to go next

The build order in the README is designed so you are always shippable:

1. ✅ this notebook = baseline + cSEBBs
2. **Frequency MixStyle sweep** — `model.mixstyle_p` in the config (0.3 / 0.5 / 0.7)
3. **Self-training** — pseudo-label the bronze tier with this model, promote confident
   predictions to silver, retrain. See `README.md`.
4. **Seed ensembling** — train 3 seeds, average frame scores, re-tune cSEBBs on the
   ensemble (re-tuning after ensembling matters; the score distribution shifts).

**Ablations worth running** (each is one config flag):

| Flag | Tests |
|---|---|
| `--no-beats` | how much BEATs is actually worth on Vaani |
| `model.n_basis: 1` | FDY vs plain CRNN — check per class, it can hurt fans/engines |
| `loss.lambda_cons: 0` | value of mean-teacher |
| `--method median` in tuning | cSEBBs vs frame thresholding |
